In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]

columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]

df_day1 = spark.createDataFrame(passengers_day1, columns)

In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
passengers_day2,
columns
)

In [0]:
df_day1.write.format("delta") \
.mode("overwrite") \
.save("/tmp/passengers_delta")

In [0]:
delta_df = spark.read.format("delta").load("/tmp/passengers_delta")

delta_df.count()

5

In [0]:
display(delta_df)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/tmp/passengers_delta")

display(deltaTable.history())

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T11:12:10.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2383279300199191),048fe09e-88ce-4831-8267-8f1fcd16dfa9,0617-110001-acftr7vp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1967)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
deltaTable.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
spark.read.format("delta")
.load("/tmp/passengers_delta")
.filter("passenger_id=102")
)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India


In [0]:
display(
spark.read.format("delta")
.load("/tmp/passengers_delta")
.filter("passenger_id=106")
)

passenger_id,passenger_name,city,travel_class,country
106,Neha Singh,Pune,Economy,India


In [0]:
version0 = spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/passengers_delta")

display(version0)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
latest = spark.read.format("delta") \
.load("/tmp/passengers_delta")

display(latest)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
print("Version0 count =", version0.count())
print("Latest count =", latest.count())

Version0 count = 5
Latest count = 7


In [0]:
display(
version0.filter("passenger_id=102")
)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,Business,India


In [0]:
display(
latest.filter("passenger_id=102")
)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India


In [0]:
display(
version0.filter("passenger_id=104")
)

passenger_id,passenger_name,city,travel_class,country
104,Sneha Patel,Delhi,Premium Economy,India


In [0]:
display(
latest.filter("passenger_id=104")
)

passenger_id,passenger_name,city,travel_class,country
104,Sneha Patel,Hyderabad,Premium Economy,India


In [0]:
%sql
OPTIMIZE delta.`/tmp/passengers_delta`

path,metrics
dbfs:/tmp/passengers_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781695492623, 1781695493028, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE delta.`/tmp/passengers_delta`
ZORDER BY (city)

path,metrics
dbfs:/tmp/passengers_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2027), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781695569174, 1781695569784, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
DELETE FROM delta.`/tmp/passengers_delta`
WHERE passenger_id = 105

num_affected_rows
1


In [0]:
display(deltaTable.history())

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-17T11:26:54.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2383279300199191),ae8f95f3-3aab-4b0a-be98-be8a8bcdcadb,0617-110001-acftr7vp-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2027, p25FileSize -> 1999, numDeletionVectorsRemoved -> 1, minFileSize -> 1999, numAddedFiles -> 1, maxFileSize -> 1999, p75FileSize -> 1999, p50FileSize -> 1999, numAddedBytes -> 1999)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-17T11:26:53.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#13330L = 105)""])",null,List(2383279300199191),ae8f95f3-3aab-4b0a-be98-be8a8bcdcadb,0617-110001-acftr7vp-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1507, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1146, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 359)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-17T11:14:49.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2383279300199191),499325cb-7925-4b7e-8514-ca51ae414a76,0617-110001-acftr7vp-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 8941, p25FileSize -> 2027, numDeletionVectorsRemoved -> 1, minFileSize -> 2027, numAddedFiles -> 1, maxFileSize -> 2027, p75FileSize -> 2027, p50FileSize -> 2027, numAddedBytes -> 2027)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-17T11:14:47.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#12259L = passenger_id#12284L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2383279300199191),499325cb-7925-4b7e-8514-ca51ae414a76,0617-110001-acftr7vp-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 6974, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3419, materializeSourceTimeMs -> 98, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1512, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1765)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T11:12:10.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2383279300199191),048fe09e-88ce-4831-8267-8f1fcd16dfa9,0617-110001-acftr7vp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1967)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
%sql
VACUUM delta.`/tmp/passengers_delta`

path
dbfs:/tmp/passengers_delta
